<a href="https://colab.research.google.com/github/hmmnyamminji/bootcampproject/blob/main/SubWay_population.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 드라이브에 저장된 폰트 등록
import matplotlib as mpl
import matplotlib.pyplot as plt # 그래프를 그리는 모듈
import matplotlib.font_manager as fm # 폰트를 관리하는 모듈

# 드라이브 내 폰트 경로
font_path = '/content/drive/MyDrive/kwu/sub_project/font/NanumGothic.ttf'

fm.fontManager.addfont(font_path)
mpl.rc('font', family='NanumGothic') # matplotlib 기본 폰트로 설정
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['font.sans-serif'] = ['NanumGothic', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False # 마이너스(-) 기호가 깨지지 않도록 유니코드 마이너스 비활성화

print("현재 폰트: ", plt.rcParams['font.family']) # 현재 적용된 폰트 이름 출력

Mounted at /content/drive
현재 폰트:  ['NanumGothic']


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import xml.etree.ElementTree as ET
import warnings # 코드 실행 중 발생하는 불필요한 경고 메시지를 숨김
warnings.filterwarnings('ignore')
import pprint
from sklearn.linear_model import LinearRegression

### 인구 혼잡도


In [ ]:
# https://data.seoul.go.kr/dataList/OA-21285/F/1/datasetView.do

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import time

API_KEY = "696c51614b67696d3736727678634c"
BASE_URL = "http://openapi.seoul.go.kr:8088"

# 5개 역
station_to_area = {
    "강남역": "강남역",
    "잠실역": "잠실역",
    "성수역": "성수역",
    "동대문역사문화공원역": "DDP(동대문디자인플라자)",
    "홍대입구역": "홍대입구역(2호선)"
}

def call_population_api(area_name: str):

    url = f"{BASE_URL}/{API_KEY}/json/citydata_ppltn/1/5/{area_name}"
    res = requests.get(url, timeout=10)
    res.raise_for_status()
    return res.json()

def extract_population_row(raw_json: dict, station: str, area_name: str, collected_at: str):

    seoul_rtd_data = raw_json.get("SeoulRtd.citydata_ppltn", [])
    citydata = seoul_rtd_data[0] if seoul_rtd_data else {}

    population_level = (
        citydata.get("AREA_CONGEST_LVL")
        or citydata.get("AREA_CONGEST_MSG")
        or citydata.get("AREA_CONGESTION_LVL")
        or None
    )

    population_min = (
        citydata.get("AREA_PPLTN_MIN")
        or citydata.get("PPLTN_MIN")
        or citydata.get("LIVE_PPLTN_MIN")
        or None
    )

    population_max = (
        citydata.get("AREA_PPLTN_MAX")
        or citydata.get("PPLTN_MAX")
        or citydata.get("LIVE_PPLTN_MAX")
        or None
    )

    return {
        "collected_at": collected_at,
        "station": station,
        "area_name": area_name,
        "population_level": population_level,
        "population_min": population_min,
        "population_max": population_max
    }

def collect_once():

    # 5개 역에 대해 현재 시점 스냅샷 1번 수집

    collected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = []

    for station, area_name in station_to_area.items():
        try:
            raw_json = call_population_api(area_name)
            row = extract_population_row(raw_json, station, area_name, collected_at)
            rows.append(row)
            print(f"[성공] {station} -> {area_name}")
        except Exception as e:
            print(f"[실패] {station} -> {area_name} | {e}")

    df = pd.DataFrame(rows)
    return df

# 1회 실행
df_now = collect_once()
df_now = df_now.replace({None: np.nan})
df_now.replace(0, np.nan, inplace=True)
df_now.fillna(method='ffill', inplace=True)
df_now

[성공] 강남역 -> 강남역
[성공] 잠실역 -> 잠실역
[성공] 성수역 -> 성수역
[성공] 동대문역사문화공원역 -> DDP(동대문디자인플라자)
[성공] 홍대입구역 -> 홍대입구역(2호선)


,collected_at,station,area_name,population_level,population_min,population_max
0,2026-04-15 11:44:26,강남역,강남역,보통,62000,64000
1,2026-04-15 11:44:26,잠실역,잠실역,여유,16000,18000
2,2026-04-15 11:44:26,성수역,성수역,여유,16000,18000
3,2026-04-15 11:44:26,동대문역사문화공원역,DDP(동대문디자인플라자),여유,5000,5500
4,2026-04-15 11:44:26,홍대입구역,홍대입구역(2호선),보통,18000,20000


In [ ]:
station_to_area = {
    "강남역": "강남역",
    "잠실역": "잠실역",
    "홍대입구역": "홍대입구역(2호선)",
    "성수역": "성수역",
    "동대문역사문화공원역": "DDP(동대문디자인플라자)"
}
